In [ ]:
import tempfile, os
from fastcore.test import test_eq

df_seawater = pd.DataFrame({
    'SMP_ID': [0, 1, 2],
    'SMP_ID_PROVIDER': ['1', '2', '3'],
    'LON': [141.0, 142.0, 143.0],
    'LAT': [37.3, 38.3, 39.3],
    'TIME': [1234, 1235, 1236],
    'NUCLIDE': [1, 2, 3],
    'VALUE': [0.1, 1.1, 2.1],
    'AREA': [2374, 2379, 2401],
    'STATION': ['A0', 'A11', 'B234']
    })

df_biota = pd.DataFrame({
    'SMP_ID': [0, 1, 2, 3],
    'SMP_ID_PROVIDER': ['ID1', 'ID2', 'ID3', 'ID4'],
    'LON': [141.0, 142.0, 143.0, 144.0],
    'LAT': [37.3, 38.3, 39.3, 40.3],
    'TIME': [1234, 1235, 1236, 1237],
    'NUCLIDE': [1, 2, 3, 3],
    'VALUE': [0.1, 1.1, 2.1, 3.1],
    'SPECIES': [1, 2, 3, 3]
    })

dfs = {'SEAWATER': df_seawater, 'BIOTA': df_biota}
attrs = {'id': '123', 'title': 'Test title', 'summary': 'Summary test'}
dest = tempfile.mktemp(suffix='.nc')

## Encode a dataset

Let's encode our test dataframes and inspect the result.

### Global attributes

The encoder copies the template's default global attributes,
then overlays the ones provided by the caller.

In [ ]:
encoder = NetCDFEncoder(dfs, dest_fname=dest, global_attrs=attrs)
encoder.encode()

with Dataset(dest, 'r', format='NETCDF4') as nc:
    test_eq(nc.id, '123')
    test_eq(nc.title, 'Test title')
    test_eq(nc.summary, 'Summary test')

In [ ]:
with Dataset(dest, 'r', format='NETCDF4') as nc:
    for grp_name in ('seawater', 'biota'):
        grp = nc.groups[grp_name]
        test_eq('id' in grp.dimensions, True)
        test_eq(grp.dimensions['id'].isunlimited(), True)
        test_eq(len(grp.dimensions['id']), len(dfs[grp_name.upper()]))